In [1]:
model_name = "vit-ragdoll"


import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=33):
    return ti(stmt, globals=globals(), number=n) * 1000 / n



BENCHMARK_REPEAT=17
df = pd.DataFrame()

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

import torch
from torch import nn
from torchvision import models
import pandas as pd

MAGIC_NUM = 7777e-5

device = torch.device("cpu:0")
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * MAGIC_NUM for k, v in model.state_dict().items()})
model = model.to(device)

df = pd.DataFrame()
!cpupower frequency-set --governor performance

Setting cpu: 0
Setting cpu: 1
Setting cpu: 2
Setting cpu: 3
Setting cpu: 4
Setting cpu: 5
Setting cpu: 6
Setting cpu: 7
Setting cpu: 8
Setting cpu: 9
Setting cpu: 10
Setting cpu: 11
Setting cpu: 12
Setting cpu: 13
Setting cpu: 14
Setting cpu: 15
Setting cpu: 16
Setting cpu: 17
Setting cpu: 18
Setting cpu: 19
Setting cpu: 20
Setting cpu: 21
Setting cpu: 22
Setting cpu: 23
Setting cpu: 24
Setting cpu: 25
Setting cpu: 26
Setting cpu: 27
Setting cpu: 28
Setting cpu: 29
Setting cpu: 30
Setting cpu: 31
Setting cpu: 32
Setting cpu: 33
Setting cpu: 34
Setting cpu: 35
Setting cpu: 36
Setting cpu: 37
Setting cpu: 38
Setting cpu: 39
Setting cpu: 40
Setting cpu: 41
Setting cpu: 42
Setting cpu: 43
Setting cpu: 44
Setting cpu: 45
Setting cpu: 46
Setting cpu: 47


In [2]:
import gc
model_file_base = "vit.mlir"
strategy = "heuristic"
ragdoll_results = []
ragdoll_throughput = []
for bs in range(1, 37):
    model_file = model_file_base + ".bs{}".format(bs)
    source_file = model_file + ".{}".format(strategy)
    target_file = source_file + ".cpu.vmfb"
    #"""
    # gen model with specified batch-size
    !ragdoll-opt {model_file_base} --ragdoll-autodiff-prepare-batch-size=batchsize={bs} > {model_file}
    
    !ragdoll-opt {model_file}  \
    --canonicalize \
    --enable-cse-in-legalizer \
    --symbol-dce \
    --ragdoll-autodiff-vjp-public-functions='strategy=heuristic' \
    --ragdoll-autodiff-vjp \
    --inline \
    --ragdoll-autodiff-inline-function-call \
    --ragdoll-initialisation \
    --eliminate-empty-tensors \
    --ragdoll-legalise-to-iree-compatibility \
    --ragdoll-raise-linalg-to-tosa \
    --ragdoll-forward-func-removal \
    --canonicalize \
    --cse > {source_file}
    #"""
    !iree-compile {source_file} \
    -o {target_file} \
    --iree-hal-target-backends=llvm-cpu \
    --iree-llvmcpu-fail-on-out-of-bounds-stack-allocation=0 \
    --iree-opt-const-eval=1 \
    --iree-opt-const-expr-hoisting=1 \
    --iree-opt-numeric-precision-reduction=1 \
    --iree-llvmcpu-target-cpu-features=host \
    --iree-llvmcpu-enable-ukernels=all \
    --iree-llvmcpu-slp-vectorization=1 \
    --iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
    --iree-llvmcpu-target-triple=x86_64-pc-linux-elf
    
    
    #ragdoll_binary = load_executable(target_file)


    #try:
        #f1 = timeit("ragdoll_binary.forward(image_np_t)") / BENCHMARK_REPEAT
        #print('ragdoll-opt1-gpu-forward in timeit: ', f1)
    print("\n\nmeasuring #", bs)
    !iree-benchmark-module \
    --module={target_file} \
    --device=local-task \
    --function=dforward \
    --input={bs}x1000xf32 \
    --batch_size={BENCHMARK_REPEAT} \
    --benchmark_repetitions=1 \
    --batch_concurrency=1 \
    --benchmark_min_time=0.1s \
    --print_statistics=true
    #"""
    b1 = ragdoll_model_benchmark(
        target_file,
        "dforward",
        [(bs, 1000)],
        device='cpu',
        warmups=15,
        repetitions=BENCHMARK_REPEAT, 
        measure_count=1)
    b1 = np.mean(b1)    
    print("measuring #", bs)
    print(b1)
    """
    b1 = timeit("ragdoll_binary.dforward(grad_np)") / BENCHMARK_REPEAT

    print("measuring #", bs)
    print(b1)
    ragdoll_results.append(b1)
    ragdoll_throughput.append(bs/np.mean(ragdoll_bench))
    """



measuring # 1
2024-03-30T07:31:49+08:00
Running /root/miniconda3/envs/albert-research-py310/lib/python3.10/site-packages/iree/_runtime_libs/iree-benchmark-module
Run on (48 X 3978.68 MHz CPU s)
CPU Caches:
  L1 Data 32 KiB (x24)
  L1 Instruction 32 KiB (x24)
  L2 Unified 512 KiB (x24)
  L3 Unified 16384 KiB (x8)
Load Average: 0.62, 1.46, 3.25
---------------------------------------------------------------------------------------------
Benchmark                                   Time             CPU   Iterations UserCounters...
---------------------------------------------------------------------------------------------
BM_dforward/process_time/real_time        141 ms          814 ms           17 items_per_second=7.07899/s
[[ iree_hal_allocator_t memory statistics ]]
  HOST_LOCAL:            0B peak /            0B allocated /            0B freed /            0B live
DEVICE_LOCAL:     23839520B peak /     23839520B allocated /     23839520B freed /            0B live
[BenchmarkResult(

In [ ]:
for bs in range(1, 27):
    dyn_model = torch.compile(model, backend="inductor")
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = dyn_model(image)
    grad = torch.randn_like(output)
    
    #torch.cuda.reset_peak_memory_stats(device=device)
    #before = torch.cuda.memory_allocated(device=device)
    print("measuring #", bs)
    #baseline_f = timeit("dyn_model(image.to(device))", 33)
    baseline_b = timeit("torch.autograd.grad(output.to(device), [image.to(device)], grad.to(device), retain_graph=True)", 333)
    #print(baseline_f)
    print(baseline_b)

    # 记录操作后的峰值内存使用情况
    #peak_memory = torch.cuda.max_memory_allocated(device=device)
    
    # 显示结果
    #print(f"Memory used before operation: {before / (1024**2):.2f} MB")
    #print(f"Peak memory usage: {peak_memory / (1024**2):.2f} MB")

In [8]:
from timeit import timeit as ti
def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n
for bs in range(1, 27):
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)
    #torch.cuda.reset_peak_memory_stats(device=device)
    #before = torch.cuda.memory_allocated(device=device)
    print("measuring #", bs)
    baseline_f = timeit("model(image)", 30)
    baseline_b = timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3)
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    #print(baseline_f)
    print(baseline_b)
    
    # 记录操作后的峰值内存使用情况
    #peak_memory = torch.cuda.max_memory_allocated(device=device)
    
    # 显示结果
    #print(f"Memory used before operation: {before / (1024**2):.2f} MB")
    #print(f"Peak memory usage: {peak_memory / (1024**2):.2f} MB")

measuring # 1
71.99569170673688
measuring # 2
101.86722719420989
measuring # 3
140.61656594276428
measuring # 4
168.35102159529924
measuring # 5
208.47214261690775
measuring # 6
236.13815754652023
measuring # 7
274.73387649903697
measuring # 8
297.0983426397045
measuring # 9
340.7025458291173
measuring # 10
357.67144554605085
measuring # 11
418.0608919511239
measuring # 12
434.0342214951913
measuring # 13
488.0899839724104
measuring # 14
512.4179118623337
measuring # 15
567.5979492564996
measuring # 16
617.5323116282622
measuring # 17
654.5441873992482
measuring # 18
679.399644335111
measuring # 19
783.2350532213846
measuring # 20
776.6547181333104
measuring # 21
839.767176968356
measuring # 22


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f3e3e0f2e00>>
Traceback (most recent call last):
  File "/root/miniconda3/envs/albert-research-py310/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 

KeyboardInterrupt



In [4]:
!sudo cpupower frequency-set --governor powersave

Setting cpu: 0
Setting cpu: 1
Setting cpu: 2
Setting cpu: 3
Setting cpu: 4
Setting cpu: 5
Setting cpu: 6
Setting cpu: 7
Setting cpu: 8
Setting cpu: 9
Setting cpu: 10
Setting cpu: 11
Setting cpu: 12
Setting cpu: 13
Setting cpu: 14
Setting cpu: 15
Setting cpu: 16
Setting cpu: 17
Setting cpu: 18
Setting cpu: 19
Setting cpu: 20
Setting cpu: 21
Setting cpu: 22
Setting cpu: 23
Setting cpu: 24
Setting cpu: 25
Setting cpu: 26
Setting cpu: 27
Setting cpu: 28
Setting cpu: 29
Setting cpu: 30
Setting cpu: 31
Setting cpu: 32
Setting cpu: 33
Setting cpu: 34
Setting cpu: 35
Setting cpu: 36
Setting cpu: 37
Setting cpu: 38
Setting cpu: 39
Setting cpu: 40
Setting cpu: 41
Setting cpu: 42
Setting cpu: 43
Setting cpu: 44
Setting cpu: 45
Setting cpu: 46
Setting cpu: 47
